# Experiment 07 — Real Data Pipeline

End-to-end walkthrough: fetch lite Planck products, plot the TT power spectrum,
run the RBLE observatory pipeline on cached WMAP data at NSIDE 64, and display
the detection report.

**Prerequisites:** Python 3.10+, Poetry, network access for first fetch.

## 1. Environment setup

From the repo root:

```bash
cd deepiri-polomni
poetry install
poetry run jupyter lab experiments/07_real_data_pipeline.ipynb
```

Optional: set a custom cache directory before fetching.

```bash
export POLOMNI_DATA_CACHE=/path/to/cache
```

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "polomni").is_dir():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams.update({"figure.figsize": (9, 5), "font.size": 11})
print(f"polomni root: {ROOT}")

## 2. Fetch lite products

Downloads Planck TT power spectrum and ΛCDM baseline theory C_l (~KB each)
plus refreshes the GWOSC GWTC catalog.

In [ ]:
from polomni.observatory.pipeline.cache import DataCache, default_cache_dir
from polomni.observatory.pipeline.catalog import LITE_PRODUCT_IDS
from polomni.observatory.pipeline.downloader import fetch_product
from polomni.observatory.pipeline.catalog import get_product
from polomni.observatory.pipeline.sources.gwosc import fetch_gwtc_events

cache = DataCache()
print(f"Cache directory: {default_cache_dir()}")
print(f"Lite product IDs: {LITE_PRODUCT_IDS}")

for pid in LITE_PRODUCT_IDS:
    product = get_product(pid)
    result = fetch_product(product, cache)
    src = "cache" if result.from_cache and not result.downloaded else "network"
    print(f"  {pid}: {result.path.name} ({result.bytes_written / 1024:.1f} KiB, {src})")

gw = fetch_gwtc_events(cache)
print(f"GWTC events cached: {gw.results_count}")

## 3. Load power spectrum — ell vs D_l

Plot Planck binned TT C_l from the cached lite product.

In [ ]:
from polomni.observatory.pipeline.sources.cosmology import load_planck_tt_power

power_path = cache.resolved_path("planck_cmb_tt_power")
assert power_path is not None, "Run fetch cell first"

ps = load_planck_tt_power(power_path)
fig, ax = plt.subplots()
ax.errorbar(
    ps.ell,
    ps.dl,
    yerr=[ps.dl - ps.dl_err_low, ps.dl_err_high - ps.dl],
    fmt="o",
    ms=3,
    label="Planck TT (binned)",
)
ax.plot(ps.ell, ps.best_fit, "k--", alpha=0.6, label="Best-fit")
ax.set_xlabel(r"Multipole $\ell$")
ax.set_ylabel(r"$D_\ell$ [$\mu$K$^2$]")
ax.set_title("Planck DR3 CMB TT Power Spectrum (lite product)")
ax.legend()
ax.set_xlim(2, ps.ell.max())
plt.tight_layout()
plt.show()
print(f"Loaded {len(ps.ell)} binned multipoles, ell range [{ps.ell.min():.0f}, {ps.ell.max():.0f}]")

## 4. Run pipeline at NSIDE 64

Fetches WMAP K-band on first run (~100 MB), downsamples to NSIDE 64,
computes RBLE signature with a 10-map null ensemble.

In [ ]:
from polomni.observatory.pipeline.processor import run_rble_pipeline

NSIDE = 64
NULLS = 10
REPORT_DIR = ROOT / "data" / "reports"

print(f"Running RBLE pipeline (NSIDE={NSIDE}, nulls={NULLS})…")
result = run_rble_pipeline(
    cache=cache,
    map_product_id="wmap_k_band",
    target_nside=NSIDE,
    null_ensemble=NULLS,
    report_dir=REPORT_DIR,
    fetch_gw=False,
)

det = result.detection
print(f"Map product : {result.map_product_id}")
print(f"NSIDE used  : {result.nside_used}")
print(f"S_RBLE      : {det.rble_score:.4f}")
print(f"Null sigma  : {det.null_sigma:.2f}")
if result.planck_lambda is not None:
    print(f"Planck Ω_Λ h² proxy: {result.planck_lambda:.5f}")
if result.report_path:
    print(f"Report path : {result.report_path}")

## 5. Display detection report

Pretty-print the report and show falsification flags.

In [ ]:
import json

from polomni.observatory.reports.detection_report import format_report

print(format_report(result.detection))

if result.report_path and result.report_path.exists():
    payload = json.loads(result.report_path.read_text())
    print("JSON report keys:", sorted(payload.keys()))
    print("Falsification flags:")
    for flag, passed in payload.get("falsification_flags", {}).items():
        status = "PASS" if passed else "FAIL"
        print(f"  {flag}: {status}")